Ce notebook contient des tests sur 2 des meilleurs modèles (GPT-oss et Phi-3) avec un prompt contenant plusieurs entrées du dictionnaire. Les mots ainsi que leur catégorie, leur définition et leur étymologie viennent du fichier Kaikki suivant : fr-extract.jsonl.gz. Le fichier étant trop lourd, on ne travaille que sur des chunks, c'est-à-dire que le modèle a, au maximum, traité 10 mots.

A la fin on obtient un csv qui ressemble beaucoup à parsed_1.tsv, mais sur qu'un seul chunk et avec les réponses de GPT-oss.

# Installation et configuration

In [ ]:
# Uniquement pour Colab, sinon télécharger Ollama https://ollama.com/download
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,431 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main 

In [ ]:
# Uniquement pour Colab
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!pip install ollama

In [ ]:
import pandas as pd
import ollama
import json

In [ ]:
!ollama pull gpt-oss
!ollama pull phi3

# Préparation du fichier jsonl

In [ ]:
df = pd.read_json("/content/fr-extract.jsonl.gz", lines=True, chunksize=10)
# beaucoup plus rapide que de lire le fichier avec with open
# le paramètre chunksize permet de couper mon fichier en morceaux de n lignes (ici 10 lignes par chunk)

In [ ]:
chunk = next(df) # next() permet de voir chunk par chunk, si je rerun ma cellule j'ai le chunk d'après
chunk.head(10) # visualisation des données dans le fichier

,word,lang_code,lang,pos,pos_title,etymology_texts,senses,forms,sounds,translations,...,hypernyms,related,proverbs,attestations,paronyms,descendants,meronyms,antonyms,abbreviations,raw_tags
20,ouvrage,fr,Français,noun,Nom commun,"[Dérivé de ouvrer, ancienne forme de œuvrer, a...",[{'glosses': ['Travail ; action de travailler....,"[{'form': 'ouvrages', 'tags': ['plural'], 'ipa...","[{'ipa': '\u.vʁaʒ\'}, {'ipa': '\u.vʁaʒ\'}, {'i...","[{'lang_code': 'af', 'lang': 'Afrikaans', 'wor...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,ouvrage,fr,Français,verb,Forme de verbe,"[Dérivé de ouvrer, ancienne forme de œuvrer, a...",[{'glosses': ['Première personne du singulier ...,"[{'form': 'j’ouvrage', 'tags': ['indicative', ...","[{'ipa': '\u.vʁaʒ\'}, {'ipa': '\u.vʁaʒ\'}, {'i...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,siège,fr,Français,noun,Nom commun,"[Du latin sediculum (« siège, chaise, banquett...",[{'glosses': ['Meuble utilisé pour s’asseoir.'...,"[{'form': 'sièges', 'tags': ['plural'], 'ipas'...","[{'ipa': '\sjɛʒ\'}, {'ipa': '[ɛ̃ sjɛʒ]', 'audi...","[{'lang_code': 'af', 'lang': 'Afrikaans', 'wor...",...,"[{'word': 'meuble', 'sense_index': 1}]","[{'word': 'assiéger'}, {'word': 'coussiège'}, ...",[{'word': 'mon siège est fait'}],[{'date': 'XIᵉ siècle'}],NaN,NaN,NaN,NaN,NaN,NaN
23,siège,fr,Français,verb,Forme de verbe,"[Du latin sediculum (« siège, chaise, banquett...",[{'glosses': ['Première personne du singulier ...,"[{'form': 'je siège', 'tags': ['indicative', '...","[{'ipa': '\sjɛʒ\'}, {'ipa': '[ɛ̃ sjɛʒ]', 'audi...",NaN,...,NaN,NaN,NaN,[{'date': 'XIᵉ siècle'}],NaN,NaN,NaN,NaN,NaN,NaN
24,chaise,fr,Français,noun,Nom commun,[De chaire par assibilation dialectale du \r\ ...,"[{'glosses': ['Siège avec dossier, sans accoud...","[{'form': 'chaises', 'tags': ['plural'], 'ipas...","[{'ipa': '\ʃɛz\'}, {'ipa': '\ʃez\'}, {'ipa': '...","[{'lang_code': 'af', 'lang': 'Afrikaans', 'wor...",...,[{'word': 'siège'}],NaN,NaN,NaN,[{'word': 'seize'}],"[{'lang_code': 'bm', 'lang': 'Bambara', 'word'...",NaN,NaN,NaN,NaN
25,fauteuil,fr,Français,noun,Nom commun,"[Du moyen français fauteuil, de l’ancien franç...","[{'glosses': ['Siège comportant des accotoirs,...","[{'form': 'fauteuils', 'tags': ['plural'], 'ip...","[{'ipa': '\fo.tœj\'}, {'ipa': '\fo.tœj\', 'rhy...","[{'lang_code': 'af', 'lang': 'Afrikaans', 'wor...",...,NaN,[{'word': 'chesterfield'}],NaN,[{'date': '1589'}],NaN,NaN,"[{'word': 'accoudoir'}, {'word': 'dossier'}, {...",NaN,NaN,NaN
26,fauteuil,nl,Néerlandais,noun,Nom commun,[Du français fauteuil.],"[{'glosses': ['Fauteuil.'], 'topics': ['furnit...",NaN,"[{'audio': 'Nl-fauteuil.ogg', 'ogg_url': 'http...",NaN,...,NaN,"[{'word': 'stoel'}, {'word': 'zitmeubel'}]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,meuble,fr,Français,noun,Nom commun,"[Du latin populaire *mŏbĭlis, altération, par ...","[{'glosses': ['Objet qui garnit, qui orne une ...","[{'form': 'meubles', 'tags': ['plural'], 'ipas...","[{'ipa': '\mœbl\'}, {'ipa': '\mœbl\', 'rhymes'...","[{'lang_code': 'af', 'lang': 'Afrikaans', 'wor...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,meuble,fr,Français,adj,Adjectif,"[Du latin populaire *mŏbĭlis, altération, par ...","[{'glosses': ['Caractère de ce qui est mou, ma...","[{'form': 'meubles', 'tags': ['plural', 'mascu...","[{'ipa': '\mœbl\'}, {'ipa': '\mœbl\', 'rhymes'...","[{'lang_code': 'en', 'lang': 'Anglais', 'word'...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[{'word': 'dur', 'sense_index': 1}, {'word': '...",NaN,NaN
29,meuble,fr,Français,verb,Forme de verbe,"[Du latin populaire *mŏbĭlis, altération, par ...",[{'glosses': ['Première personne du singulier ...,"[{'form': 'je meuble', 'tags': ['indicative', ...","[{'ipa': '\mœbl\'}, {'ipa': '\mœbl\', 'rhymes'...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
small_df = chunk[chunk["lang"] == "Français"][["word", "pos_title", "senses", "etymology_texts"]]
# on ne prend que les mots français du chunk et on fait apparaître les colonnes word, pos_title, senses et etymology_texts
# on aura besoin de ces quatre colonnes pour créer notre json_entry;
# word correspond au mot
# pos_title correspond à la catégorie
# senses correspond à la définition
# etymology_texts correspond à l'étymologie
small_df["senses"] = small_df["senses"].apply( lambda x :  x[0]["glosses"] ) # je ne prends que la première gloss
small_df = small_df.rename(columns={"word": "mot", "pos_title": "catégorie", "senses": "définition", "etymology_texts": "étymologie"})
# je renomme les colonnes pour que ça corresponde à mon fichier json_entry (et pour que ce soit en français)
small_df # df à concat avec le df de la fin (incluant les colonnes lang, base, pref, suff, comps, type)

,mot,catégorie,définition,étymologie
20,ouvrage,Nom commun,[Travail ; action de travailler.],"[Dérivé de ouvrer, ancienne forme de œuvrer, a..."
21,ouvrage,Forme de verbe,[Première personne du singulier de l’indicatif...,"[Dérivé de ouvrer, ancienne forme de œuvrer, a..."
22,siège,Nom commun,[Meuble utilisé pour s’asseoir.],"[Du latin sediculum (« siège, chaise, banquett..."
23,siège,Forme de verbe,[Première personne du singulier du présent de ...,"[Du latin sediculum (« siège, chaise, banquett..."
24,chaise,Nom commun,"[Siège avec dossier, sans accoudoirs.]",[De chaire par assibilation dialectale du \r\ ...
25,fauteuil,Nom commun,"[Siège comportant des accotoirs, et un dossier.]","[Du moyen français fauteuil, de l’ancien franç..."
27,meuble,Nom commun,"[Objet qui garnit, qui orne une chambre, un ap...","[Du latin populaire *mŏbĭlis, altération, par ..."
28,meuble,Adjectif,"[Caractère de ce qui est mou, malléable.]","[Du latin populaire *mŏbĭlis, altération, par ..."
29,meuble,Forme de verbe,[Première personne du singulier du présent de ...,"[Du latin populaire *mŏbĭlis, altération, par ..."
32,mardi,Nom commun,"[Deuxième jour de la semaine, qui suit le lund...",[Du latin Martis dies signifiant le jour de Ma...


In [ ]:
# besoin de créer une liste de toutes ces informations pour créer un fichier jsonl
# on utilise "records" pour avoir le nom des colonnes comme clés
list_of_dict = small_df.to_dict("records")
list_of_dict

[{'mot': 'ouvrage',
  'catégorie': 'Nom commun',
  'définition': ['Travail ; action de travailler.'],
  'étymologie': ['Dérivé de ouvrer, ancienne forme de œuvrer, avec le suffixe -age.']},
 {'mot': 'ouvrage',
  'catégorie': 'Forme de verbe',
  'définition': ['Première personne du singulier de l’indicatif présent du verbe ouvrager.'],
  'étymologie': ['Dérivé de ouvrer, ancienne forme de œuvrer, avec le suffixe -age.']},
 {'mot': 'siège',
  'catégorie': 'Nom commun',
  'définition': ['Meuble utilisé pour s’asseoir.'],
  'étymologie': ['Du latin sediculum (« siège, chaise, banquette ») diminutif de sedes, devenu en bas latin *sedicum de là siège. Il est lié au verbe latin sedere (« asseoir »).']},
 {'mot': 'siège',
  'catégorie': 'Forme de verbe',
  'définition': ['Première personne du singulier du présent de l’indicatif de siéger.'],
  'étymologie': ['Du latin sediculum (« siège, chaise, banquette ») diminutif de sedes, devenu en bas latin *sedicum de là siège. Il est lié au verbe lati

In [ ]:
with open("chunk.jsonl", "w", encoding="utf-8") as f :
       for d in list_of_dict :
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

In [ ]:
json_entries = []

with open("chunk.jsonl", "r", encoding="utf-8") as f :
    for line in f :
        json_entry = json.loads(line)
        json_entries.append(json_entry)

# Tests

## Sur GPT-oss

In [ ]:
llm_responses = []

SYSTEM_PROMPT = """Tu es un assistant qui répond efficacement aux questions
à partir d'un fichier JSON qui contient plusieurs informations à propos d'un mot.
Tu ne dois pas donner d'explications, tu ne dois pas raisonner, tu ne dois pas reformuler la tâche."""

for json_entry in json_entries :
  prompt = f"""l'enregistrement en JSON suivant : {json_entry} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
      * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
      * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
      * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou NULL si l'information est absente)
      * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou NULL si l'information est absente)
      * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (liste de composants ou NULL si l'information est absente)
      * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou NULL si l'information est absente)"""

  response = ollama.chat(
          model="gpt-oss",
          messages=[
              {
                  "role": "system",
                  "content": SYSTEM_PROMPT
              },
              {"role": "user", "content": prompt},
          ],
          format="json"
      )
  print(f"Réponse : {response['message']['content']}")

  content = response["message"]["content"]
  data = json.loads(content)
  llm_responses.append(data)

Réponse : {
  "Q1": "français",
  "Q2": "accueillir",
  "Q3": null,
  "Q4": null,
  "Q5": ["accueillir"],
  "Q6": "déverbal"
}
Réponse : {
  "Q1": "latin",
  "Q2": "lĕgĕre",
  "Q3": null,
  "Q4": null,
  "Q5": null,
  "Q6": null
}
Réponse : {
  "Q1": "italien",
  "Q2": "lira",
  "Q3": null,
  "Q4": null,
  "Q5": null,
  "Q6": null
}
Réponse : {"Q1":"Latin","Q2":"encyclopaedia","Q3":"ἐγκύκλο","Q4":null,"Q5":["ἐγκύκλιος","παιδεία"],"Q6":"composition"}
Réponse : {"Q1":"latin","Q2":"encyclopaedia","Q3":"en","Q4":"ia","Q5":["ἐγκύκλιος","παιδεία"],"Q6":"composition"}
Réponse : {"Q1":"japonais","Q2":"manga","Q3":null,"Q4":null,"Q5":null,"Q6":null}


In [ ]:
df_llm_responses = pd.DataFrame(llm_responses)
df_llm_responses_modified = df_llm_responses.rename(columns={"Q1": "lang", "Q2": "base", "Q3": "pref", "Q4": "suff", "Q5": "comps", "Q6": "type"})
df_llm_responses_modified

,lang,base,pref,suff,comps,type
0,français,accueillir,None,None,[accueillir],déverbal
1,latin,lĕgĕre,None,None,None,None
2,italien,lira,None,None,None,None
3,Latin,encyclopaedia,ἐγκύκλο,None,"[ἐγκύκλιος, παιδεία]",composition
4,latin,encyclopaedia,en,ia,"[ἐγκύκλιος, παιδεία]",composition
5,japonais,manga,None,None,None,None


In [ ]:
small_df_modified = small_df.rename(columns={"mot": "lemma", "catégorie": "cat", "définition": "def", "étymologie": "etym"}).reset_index(drop = True)
full_df = pd.concat([small_df_modified, df_llm_responses_modified], axis=1)
full_df

,lemma,cat,def,etym,lang,base,pref,suff,comps,type
0,accueil,Nom commun,[Cérémonie ou prestation réservée à un nouvel ...,[Déverbal de accueillir.],français,accueillir,None,None,[accueillir],déverbal
1,lire,Verbe,[Interpréter des informations écrites sous for...,"[Du latin lĕgĕre (« id. »), proprement « recue...",latin,lĕgĕre,None,None,None,None
2,lire,Nom commun,[Monnaie utilisée en Italie avant l’usage de l...,"[De l’italien lira, du latin libra (« livre »,...",italien,lira,None,None,None,None
3,encyclopédie,Nom commun,"[Ensemble, enchaînement de toutes les connaiss...",[Du latin encyclopaedia forgé à la Renaissance...,Latin,encyclopaedia,ἐγκύκλο,None,"[ἐγκύκλιος, παιδεία]",composition
4,encyclopédie,Forme de verbe,[Première personne du singulier de l’indicatif...,[Du latin encyclopaedia forgé à la Renaissance...,latin,encyclopaedia,en,ia,"[ἐγκύκλιος, παιδεία]",composition
5,manga,Nom commun,"[Bande dessinée japonaise, souvent en noir et ...","[Du japonais 漫画, manga (de même sens).]",japonais,manga,None,None,None,None


In [ ]:
full_df.to_csv("tests_gpt_oss.csv", index=False)

## Sur Phi-3

In [ ]:
SYSTEM_PROMPT = """Tu es un assistant qui répond efficacement aux questions
à partir d'un fichier JSON qui contient plusieurs informations à propos d'un mot.
Tu ne dois pas donner d'explications, tu ne dois pas raisonner, tu ne dois pas reformuler la tâche."""

for json_entry in json_entries :
  prompt = f"""l'enregistrement en JSON suivant : {json_entry} contient un mot du français, une catégorie, une définition et une étymologie. Construis un nouvel enregistrement JSON qui contient les réponses aux questions suivantes :
      * Q1 = quelle est la langue d'origine du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
      * Q2 = quelle est la base du mot qui est indiquée dans l'étymologie ? (un seul mot ou NULL si l'information est absente)
      * Q3 = si un préfixe est indiqué dans l'étymologie, quel est ce préfixe ? (un seul mot ou NULL si l'information est absente)
      * Q4 = si un suffixe est indiqué dans l'étymologie, quel est ce suffixe ? (un seul mot ou NULL si l'information est absente)
      * Q5 = si des composants sont indiqués dans l'étymologie, quels sont ces composants ? (liste de composants ou NULL si l'information est absente)
      * Q6 = si le type du procédé morphologique est indiqué dans l'étymologie (suffixation, préfixation, composition, conversion, apocope, etc.), quel est ce type ? (un seul mot ou NULL si l'information est absente)"""

  response = ollama.chat(
          model="phi3",
          messages=[
              {
                  "role": "system",
                  "content": SYSTEM_PROMPT
              },
              {"role": "user", "content": prompt},
          ],
          format="json"
      )
  print(f"Réponse : {response['message']['content']}")

# Conclusions

GPT-oss a mieux compris la tâche : il m'a sorti des enregistrements json pour chaque entrée. Phi-3 est catastrophique : il a raisonné, a inventé des entrées qui n'étaient pas dans le prompt, les sorties ne sont pas toutes en json.

Dans l'ensemble les deux modèles ne sont pas terribles. Même si GPT-oss a bien compris la tâche, les réponses sont en partie mauvaises. Si on compare avec le tsv parsed_1, pour le lemma accueil, Nabil a trouvé que c'était une conversion alors que moi il a dit "déverbal".

**Il faut tester un meilleur modèle sur Grid5k.**